# Replication on Kaggle

Runs `scripts/replicate.py` of https://github.com/Tachia/cdpr-fault-tolerant on a Kaggle CPU machine,
as described in `REPLICATION.md`. Settings of the notebook: **Accelerator: None**, **Internet: on**.
Run it with *Save Version → Save & Run All*, so that it runs in the background and its output is kept.

The report is written to `/kaggle/working/replication_runs/<label>/`. Download that folder from the
*Output* tab and compare it with the other machines (`scripts/compare_replications.py`).

In [ ]:
# Settings
TAG = "v1.0-replication"   # the commit every replication must use (REPLICATION.md, Section 2)
LABEL = "kaggle"
TIER = 2                   # 1: stored results (~30 min); 2: + sample simulated again (~2 h); 3: + whole campaign
MAX_CAMPAIGN_JOBS = 2800   # tier 3 only: new campaign jobs per session, so that a session ends in time
TORCH_CPU = False          # True installs the smaller CPU build of torch instead of the reference wheel

import os, subprocess
REPO = "/tmp/cdpr-fault-tolerant"
OUT = "/kaggle/working"
print(os.cpu_count(), "cores")

## 1. Code and environment

The repository is cloned outside `/kaggle/working`, so that the environment (several GB) is not
saved as output. The folders that must survive a session (the report, the compiled solvers and,
for tier 3, the campaign) are linked into `/kaggle/working`.

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", TAG, "--depth", "1",
                    "https://github.com/Tachia/cdpr-fault-tolerant.git", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
# Continue from an earlier session: attach the output of the earlier version of this notebook as
# input (Add Input → Your Work), then its folders are copied back here.
import glob, shutil
for name in ("replication_runs", "build", "phase5_work"):
    earlier = sorted(glob.glob(f"/kaggle/input/*/{name}"))
    if earlier and not os.path.exists(f"{OUT}/{name}"):
        shutil.copytree(earlier[0], f"{OUT}/{name}", symlinks=True)
        print("continued from", earlier[0])
for name, inside in (("replication_runs", "replication/runs"), ("build", "build"),
                     ("phase5_work", "results/phase5_work")):
    os.makedirs(f"{OUT}/{name}", exist_ok=True)
    target = f"{REPO}/{inside}"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.islink(target):
        if os.path.isdir(target):
            shutil.rmtree(target)
        os.symlink(f"{OUT}/{name}", target)
# git ignores these folders, but not links with their names; list the links in the local exclude file
with open(f"{REPO}/.git/info/exclude", "a") as f:
    f.write("replication/runs\nbuild\nresults/phase5_work\n")

In [ ]:
args = ["bash", f"{REPO}/replication/setup_env.sh"] + (["--torch-cpu"] if TORCH_CPU else [])
subprocess.run(args, check=True)

## 2. Replication

In [ ]:
cmd = [f"{REPO}/.venv/bin/python", "scripts/replicate.py", "--label", LABEL, "--tier", str(TIER),
       "--jobs", str(os.cpu_count())]
if TIER == 3:
    cmd += ["--max-campaign-jobs", str(MAX_CAMPAIGN_JOBS)]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()

In [ ]:
print(open(f"{OUT}/replication_runs/{LABEL}/report.md").read())

## 3. Output

`/kaggle/working/replication_runs/<label>/` holds the report. For tier 3, if the report says
UNFINISHED, run a new version of this notebook with the output of this one attached as input; it
continues where this one stopped.